## Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

In [15]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model(model = "groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000165B4A0CEC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000165B4A0DD90>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [21]:
from pydantic import BaseModel, Field 

class Movie(BaseModel):
    title: str = Field(description = "The title of the movie")
    year: int = Field(description = "This year the movie was released")
    director: str = Field(description = "The director of the movie")
    rating: float = Field(description = "The movies rating out of 10")

In [22]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000165B4A0CEC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000165B4A0DD90>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out

In [23]:
response = model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [25]:
# without structured output

response = model.invoke("Provide details about the moview Inception")
print(response.content)

<think>
Okay, the user is asking for details about the movie Inception. Let me start by recalling what I know about it. Directed by Christopher Nolan, released in 2010, right? The main actor is Leonardo DiCaprio. The story revolves around dreams within dreams. I need to structure the information properly.

First, the plot. The main character is Dom Cobb, a thief who enters people's dreams to steal secrets. His team wants to perform an inception, which is planting an idea instead. The concept of layers in dreams is key here. Each layer is deeper, and time slows down. Then there's the idea of limbo, a place where the subconscious is uncontrolled. The team has to navigate these layers to plant the idea.

Characters: Leonardo DiCaprio as Cobb, Joseph Gordon-Levitt as Arthur, the architect. Ellen Page plays Ariadne, the architect. There's Tom Hardy as Eames, and Tom Berenger as the projection of the target. The target is Robert Fischer, played by Cillian Murphy. Also, Marion Cotillard as Ma

### Message output alongside parsed structure

In [28]:
model_with_structure = model.with_structured_output(Movie, include_raw = True)
response = model_with_structure.invoke("Provide details about the moview Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see what I need to do here. The tools provided include a Movie function with parameters like director, rating, title, and year. All these parameters are required.\n\nFirst, I need to recall the information about Inception. The title is obviously "Inception". The director is Christopher Nolan. It was released in 2010, so the year is 2010. The rating, probably IMDb, is around 8.8. \n\nWait, the user might expect the rating out of 10. Let me confirm the exact rating. IMDb lists it as 8.8/10. So I should use that. The year is definitely 2010. The director is Christopher Nolan. \n\nI need to structure this into the function call. The required fields are title, year, director, and rating. All are present here. Let me check the types: title and director are strings, year is an integer, rating is a number. Everything looks good. \n\nNo other functions ar

### Nested Structure

In [29]:
class Actor(BaseModel):
    name: str 
    role: str 

class MovieDetails(BaseModel):
    title: str 
    year: int 
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description = "Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role="Dominick 'Dom' Cobb"), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role="Cobb's Forged Identity")], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

In [38]:
# using create agent 
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format = ContactInfo
)

agent.invoke({
    "messages": [{
        "role": "user", 
        "content" : "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
    }]
})

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='85f2d79d-d1d9-4a41-83b6-b03a2d24b39c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given string: "John Doe, john@example.com, (555) 123-4567". The tools provided include a ContactInfo function that requires name, email, and phone number.\n\nFirst, I need to parse the input. The name is "John Doe", which is straightforward. The email is "john@example.com". The phone number is "(555) 123-4567". I should check if the phone number is in the correct format. The function parameters don\'t specify formatting, so I\'ll assume the given format is acceptable. All required fields are present, so I can call the ContactInfo function with these details.\n', 'tool_calls': [{'id': 'h15nkvxa6', 'function': {'arguments': '{"email":"john@example.com","nam

### TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [33]:
from typing_extensions import TypedDict, Annotated 

class MovieDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[str, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': '2012'}

In [34]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Adventure'],
 'title': 'Avengers',
 'year': 2012}

### DataClasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [39]:
from dataclasses import dataclass 

@dataclass 
class ContactInfo:
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format = ContactInfo
)

agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
    }]
})

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='acd009a0-5762-46ab-97ef-48b1e2544793'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given string: "John Doe, john@example.com, (555) 123-4567". The tools provided include a ContactInfo function that requires name, email, and phone. \n\nFirst, I need to parse the input. The name is "John Doe", which is straightforward. The email is "john@example.com". The phone number is "(555) 123-4567". I should check if the phone number is in the correct format. The function parameters require all three fields, so I need to make sure each is correctly identified. \n\nI don\'t see any additional information here, so it\'s just a matter of splitting the components. The name is first, then email, then phone. The phone number includes parentheses and a spa